# Inventory Planning: Safety Stock and Reorder Point

This notebook translates demand forecasts and variability into inventory replenishment decisions using safety stock and reorder point calculations.


## Business Objective

The objective is to determine appropriate safety stock and reorder points for high-volume SKUs under different service-level targets, balancing stockout risk against inventory holding cost.


## Inputs from Forecasting

The following inputs are derived from the forecasting analysis:
- Average daily demand
- Demand standard deviation
- Coefficient of variation
- Selected forecasting model per SKU


## Inventory Policy Assumptions

- Continuous review (reorder point) policy
- Daily inventory review
- Fixed lead time of 7 days
- Independent daily demand


## Service Levels Considered

The following service levels are evaluated:
- 90%
- 95%
- 98%

Higher service levels increase safety stock and holding cost but reduce stockout risk.


## Inventory Planning Formulas

Mean demand during lead time:
μ_L = μ × L

Standard deviation during lead time:
σ_L = σ × √L

Safety Stock:
Safety Stock = z × σ_L

Reorder Point (ROP):
ROP = μ_L + Safety Stock


## Notebook Roadmap

1. Load forecasting outputs
2. Define lead time and service-level parameters
3. Calculate safety stock for each SKU
4. Calculate reorder points
5. Compare inventory implications across service levels
6. Export recommended reorder parameters


In [1]:
import pandas as pd
import numpy as np

FORECAST_PATH = "../outputs/forecasts/forecast_accuracy_by_sku.csv"


In [2]:
fc = pd.read_csv(FORECAST_PATH)
fc.shape


(50, 9)

In [3]:
fc.columns, fc.head(3)


(Index(['series_id', 'model_baseline', 'smape_baseline', 'avg_daily_demand',
        'std_daily_demand', 'cv', 'model_ets', 'smape_ets', 'better_model'],
       dtype='object'),
                      series_id  model_baseline  smape_baseline  \
 0  FOODS_1_085_CA_1_validation  seasonal_naive        0.520839   
 1  FOODS_1_218_CA_1_validation  seasonal_naive        0.775914   
 2  FOODS_2_019_CA_1_validation  seasonal_naive        0.508236   
 
    avg_daily_demand  std_daily_demand        cv model_ets  smape_ets  \
 0         10.641924          6.947792  0.652870       ETS   0.478024   
 1         11.702561          9.325353  0.796864       ETS   0.617933   
 2         18.465238         10.984678  0.594884       ETS   0.400250   
 
   better_model  
 0          ETS  
 1          ETS  
 2          ETS  )

## Inventory Parameters

Lead time and service-level assumptions are defined to evaluate safety stock and reorder points under different inventory risk targets.


In [4]:
LEAD_TIME_DAYS = 7


In [5]:
service_levels = {
    "90%": 0.90,
    "95%": 0.95,
    "98%": 0.98
}


In [6]:
from scipy.stats import norm

z_scores = {k: norm.ppf(v) for k, v in service_levels.items()}
z_scores


{'90%': 1.2815515655446004,
 '95%': 1.6448536269514722,
 '98%': 2.0537489106318225}

## Safety Stock and Reorder Point Calculation

Safety stock and reorder points are calculated for each SKU under different service-level targets using demand variability and lead time assumptions.


In [7]:
inv_base = fc.copy()

# Mean demand during lead time
inv_base["mean_demand_LT"] = inv_base["avg_daily_demand"] * LEAD_TIME_DAYS

# Std dev of demand during lead time
inv_base["std_demand_LT"] = inv_base["std_daily_demand"] * np.sqrt(LEAD_TIME_DAYS)

inv_base[[
    "series_id",
    "avg_daily_demand",
    "std_daily_demand",
    "mean_demand_LT",
    "std_demand_LT"
]].head()


,series_id,avg_daily_demand,std_daily_demand,mean_demand_LT,std_demand_LT
0,FOODS_1_085_CA_1_validation,10.641924,6.947792,74.493466,18.382131
1,FOODS_1_218_CA_1_validation,11.702561,9.325353,81.917930,24.672566
2,FOODS_2_019_CA_1_validation,18.465238,10.984678,129.256665,29.062726
3,FOODS_2_197_CA_1_validation,21.274438,12.736485,148.921066,33.697572
4,FOODS_2_244_CA_1_validation,10.476215,6.920014,73.333508,18.308636


In [8]:
records = []

for _, row in inv_base.iterrows():
    for sl, z in z_scores.items():
        safety_stock = z * row["std_demand_LT"]
        rop = row["mean_demand_LT"] + safety_stock

        records.append({
            "series_id": row["series_id"],
            "service_level": sl,
            "z_score": z,
            "avg_daily_demand": row["avg_daily_demand"],
            "std_daily_demand": row["std_daily_demand"],
            "lead_time_days": LEAD_TIME_DAYS,
            "safety_stock": safety_stock,
            "reorder_point": rop
        })

inventory_params = pd.DataFrame(records)
inventory_params.head()


,series_id,service_level,z_score,avg_daily_demand,std_daily_demand,lead_time_days,safety_stock,reorder_point
0,FOODS_1_085_CA_1_validation,90%,1.281552,10.641924,6.947792,7,23.557648,98.051114
1,FOODS_1_085_CA_1_validation,95%,1.644854,10.641924,6.947792,7,30.235914,104.729380
2,FOODS_1_085_CA_1_validation,98%,2.053749,10.641924,6.947792,7,37.752281,112.245747
3,FOODS_1_218_CA_1_validation,90%,1.281552,11.702561,9.325353,7,31.619166,113.537096
4,FOODS_1_218_CA_1_validation,95%,1.644854,11.702561,9.325353,7,40.582760,122.500690


In [9]:
inventory_params.groupby("service_level")[["safety_stock", "reorder_point"]].mean()


,safety_stock,reorder_point
service_level,,
90%,47.420079,168.277402
95%,60.863012,181.720336
98%,75.992990,196.850314


## Stockout vs Holding Cost Tradeoff (Simulation)

A simple simulation is used to compare service levels by estimating:
- Stockout days (days demand exceeds available inventory before replenishment arrives)
- Average on-hand inventory (proxy for holding cost)
This supports a practical service-level recommendation per SKU.


In [10]:
# Minimal reload for simulation (top 50 SKUs only)
sales = pd.read_csv("../data/raw/sales_train_validation.csv")
calendar = pd.read_csv("../data/raw/calendar.csv")

STORE_ID = "CA_1"
demand_cols = [c for c in sales.columns if c.startswith("d_")]

sales_store = sales[sales["store_id"] == STORE_ID].copy()
sales_store["total_demand"] = sales_store[demand_cols].sum(axis=1)

top50_ids = (
    sales_store.sort_values("total_demand", ascending=False)
    .head(50)["id"]
    .tolist()
)

top50 = sales_store[sales_store["id"].isin(top50_ids)].drop(columns=["total_demand"])

demand_long2 = top50.melt(
    id_vars=["id"],
    value_vars=demand_cols,
    var_name="d",
    value_name="demand"
).merge(calendar[["d", "date"]], on="d", how="left")

demand_long2["date"] = pd.to_datetime(demand_long2["date"])
demand_long2 = demand_long2.sort_values(["id", "date"])

demand_long2.head()


,id,d,demand,date
3,FOODS_1_085_CA_1_validation,d_1,10,2011-01-29
53,FOODS_1_085_CA_1_validation,d_2,15,2011-01-30
103,FOODS_1_085_CA_1_validation,d_3,13,2011-01-31
153,FOODS_1_085_CA_1_validation,d_4,5,2011-02-01
203,FOODS_1_085_CA_1_validation,d_5,11,2011-02-02


In [11]:
SIM_DAYS = 365

sim_data = (
    demand_long2.groupby("id")
    .tail(SIM_DAYS)
    .copy()
)

sim_data["t"] = sim_data.groupby("id").cumcount()
sim_data.head()


,id,d,demand,date,t
77403,FOODS_1_085_CA_1_validation,d_1549,5,2015-04-26,0
77453,FOODS_1_085_CA_1_validation,d_1550,3,2015-04-27,1
77503,FOODS_1_085_CA_1_validation,d_1551,3,2015-04-28,2
77553,FOODS_1_085_CA_1_validation,d_1552,3,2015-04-29,3
77603,FOODS_1_085_CA_1_validation,d_1553,1,2015-04-30,4


In [12]:
HOLDING_COST_PER_UNIT_PER_DAY = 0.02   # proxy cost per unit held per day
STOCKOUT_PENALTY_PER_UNIT = 1.0        # penalty per unit short (lost sales/backorder)


In [13]:
def simulate_rop_policy(demands, rop, mean_lt_demand, lead_time):
    S = rop + mean_lt_demand  # order-up-to level
    
    on_hand = S
    pipeline = []  # list of (arrival_day, qty)
    
    stockout_units = 0
    holding_units = 0
    
    for day, d in enumerate(demands):
        # Receive orders arriving today
        arrivals = [q for (arr_day, q) in pipeline if arr_day == day]
        if arrivals:
            on_hand += sum(arrivals)
        pipeline = [(arr_day, q) for (arr_day, q) in pipeline if arr_day != day]
        
        # Demand happens
        if on_hand >= d:
            on_hand -= d
        else:
            stockout_units += (d - on_hand)
            on_hand = 0
        
        # Holding accumulation
        holding_units += on_hand
        
        # Inventory position
        on_order = sum(q for _, q in pipeline)
        inventory_position = on_hand + on_order
        
        # Reorder trigger
        if inventory_position <= rop:
            order_qty = max(0, S - inventory_position)
            pipeline.append((day + lead_time, order_qty))
    
    avg_on_hand = holding_units / len(demands)
    return stockout_units, avg_on_hand


In [14]:
sim_results = []

for sku_id, df in sim_data.groupby("id"):
    demands = df["demand"].values
    
    sku_params = inventory_params[inventory_params["series_id"] == sku_id]
    
    for sl in sku_params["service_level"].unique():
        row = sku_params[sku_params["service_level"] == sl].iloc[0]
        
        rop = row["reorder_point"]
        mean_lt = row["avg_daily_demand"] * LEAD_TIME_DAYS
        
        stockout_units, avg_on_hand = simulate_rop_policy(
            demands=demands,
            rop=rop,
            mean_lt_demand=mean_lt,
            lead_time=LEAD_TIME_DAYS
        )
        
        holding_cost = avg_on_hand * HOLDING_COST_PER_UNIT_PER_DAY * len(demands)
        stockout_cost = stockout_units * STOCKOUT_PENALTY_PER_UNIT
        total_cost = holding_cost + stockout_cost
        
        sim_results.append({
            "series_id": sku_id,
            "service_level": sl,
            "stockout_units": stockout_units,
            "avg_on_hand": avg_on_hand,
            "holding_cost_proxy": holding_cost,
            "stockout_cost_proxy": stockout_cost,
            "total_cost_proxy": total_cost
        })

sim_results = pd.DataFrame(sim_results)
sim_results.head()


,series_id,service_level,stockout_units,avg_on_hand,holding_cost_proxy,stockout_cost_proxy,total_cost_proxy
0,FOODS_1_085_CA_1_validation,90%,31.455420,73.391670,535.759193,31.455420,567.214613
1,FOODS_1_085_CA_1_validation,95%,24.777154,79.941860,583.575577,24.777154,608.352731
2,FOODS_1_085_CA_1_validation,98%,17.260787,87.293825,637.244920,17.260787,654.505707
3,FOODS_1_218_CA_1_validation,90%,0.000000,90.564615,661.121686,0.000000,661.121686
4,FOODS_1_218_CA_1_validation,95%,0.000000,99.528209,726.555923,0.000000,726.555923


In [15]:
best_policy = (
    sim_results.sort_values(["series_id", "total_cost_proxy"])
    .groupby("series_id")
    .head(1)
    .reset_index(drop=True)
)

best_policy["service_level"].value_counts()


service_level
90%    40
98%     7
95%     3
Name: count, dtype: int64

In [16]:
# Join best policy with the corresponding ROP + Safety Stock values
best_with_params = best_policy.merge(
    inventory_params,
    on=["series_id", "service_level"],
    how="left"
)

# Add forecast performance + variability from fc
best_with_params = best_with_params.merge(
    fc[["series_id", "smape_baseline", "smape_ets", "better_model", "cv", "avg_daily_demand", "std_daily_demand"]],
    on="series_id",
    how="left"
)

best_with_params.head()


,series_id,service_level,stockout_units,avg_on_hand,holding_cost_proxy,stockout_cost_proxy,total_cost_proxy,z_score,avg_daily_demand_x,std_daily_demand_x,lead_time_days,safety_stock,reorder_point,smape_baseline,smape_ets,better_model,cv,avg_daily_demand_y,std_daily_demand_y
0,FOODS_1_085_CA_1_validation,90%,31.45542,73.391670,535.759193,31.45542,567.214613,1.281552,10.641924,6.947792,7,23.557648,98.051114,0.520839,0.478024,ETS,0.652870,10.641924,6.947792
1,FOODS_1_218_CA_1_validation,90%,0.00000,90.564615,661.121686,0.00000,661.121686,1.281552,11.702561,9.325353,7,31.619166,113.537096,0.775914,0.617933,ETS,0.796864,11.702561,9.325353
2,FOODS_2_019_CA_1_validation,90%,0.00000,130.240903,950.758595,0.00000,950.758595,1.281552,18.465238,10.984678,7,37.245382,166.502047,0.508236,0.400250,ETS,0.594884,18.465238,10.984678
3,FOODS_2_197_CA_1_validation,90%,0.00000,156.323200,1141.159358,0.00000,1141.159358,1.281552,21.274438,12.736485,7,43.185177,192.106243,0.591558,0.758981,Seasonal Naive,0.598676,21.274438,12.736485
4,FOODS_2_244_CA_1_validation,90%,0.00000,88.840066,648.532478,0.00000,648.532478,1.281552,10.476215,6.920014,7,23.463461,96.796969,0.859070,0.532942,ETS,0.660545,10.476215,6.920014


In [19]:
# 1) SKU-policy recommendation table (main Power BI table)
best_with_params.to_csv(
    "../outputs/inventory_planning/recommended_reorder_policy.csv",
    index=False
)

# 2) Full simulation results (optional detail page in Power BI)
sim_results.to_csv(
    "../outputs/inventory_planning/simulation_results_all_service_levels.csv",
    index=False
)

# 3) Inventory parameters for all service levels (optional detail)
inventory_params.to_csv(
    "../outputs/inventory_planning/inventory_params_all_service_levels.csv",
    index=False
)

print("Saved 3 files to outputs/inventory_planning/")


Saved 3 files to outputs/inventory_planning/
